In [ ]:
import pandas as pd
import numpy as np
import re, os
from pathlib import Path

# ============================================================
# User paths
# ============================================================

DATA_PATH = Path(r"C:\Users\oozgen\Desktop\phd\dataset\training\SOD1_cleaned_v3.csv")

OUTPUT_DIR = DATA_PATH.parent

PROCESSED_OUT = OUTPUT_DIR / "SOD1_process_v1.csv"
POSITION_OUT = OUTPUT_DIR / "SOD1_position_level_annotations_v1.csv"
LINKAGE_OUT = OUTPUT_DIR / "SOD1_linkage_level_annotations_v1.csv"
VALIDATION_OUT = OUTPUT_DIR / "SOD1_validation_issues_v1.csv"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "SOD1_process_v1_data_dictionary.csv"


# ============================================================
# Load data
# ============================================================

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

required_cols = [
    "ISIS",
    "Sequence",
    "Chemical_Pattern",
    "Linkage",
    "Linkage_Location",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Add a stable row identifier in case ISIS is ever duplicated
df = df.reset_index(drop=True)
df["row_id"] = np.arange(len(df))

# Normalize string columns
df["Sequence"] = df["Sequence"].astype(str).str.upper().str.strip()
df["Chemical_Pattern"] = df["Chemical_Pattern"].astype(str).str.strip()
df["Linkage"] = df["Linkage"].astype(str).str.strip()
df["Linkage_Location"] = df["Linkage_Location"].astype(str).str.strip()

print("Loaded shape:", df.shape)
df.head()

In [ ]:
# 1. Recalculate and validate length-derived fields

In [ ]:
# ============================================================
# 1. Recalculate and validate length-derived fields
# ============================================================

# Preserve original seq_length if present
if "seq_length" in df.columns:
    df["seq_length_original"] = df["seq_length"]

df["seq_length_recalc"] = df["Sequence"].str.len()
df["chem_pattern_length"] = df["Chemical_Pattern"].str.len()

# Overwrite seq_length with recalculated value
df["seq_length"] = df["seq_length_recalc"]

# Valid symbols
valid_bases = set("ACGT")
valid_pattern_symbols = set("MCd")  # M = MOE, C = cEt, d = DNA

df["valid_sequence"] = df["Sequence"].apply(lambda x: set(x).issubset(valid_bases))
df["valid_chemical_pattern"] = df["Chemical_Pattern"].apply(
    lambda x: set(x).issubset(valid_pattern_symbols)
)

df["lengths_match"] = (
    (df["seq_length_recalc"] == df["chem_pattern_length"]) &
    df["valid_sequence"] &
    df["valid_chemical_pattern"]
)

# Identify length or symbol issues
validation_cols = [
    "row_id",
    "ISIS",
    "Sequence",
    "Chemical_Pattern",
    "seq_length",
    "seq_length_recalc",
    "chem_pattern_length",
    "valid_sequence",
    "valid_chemical_pattern",
    "lengths_match",
]

if "seq_length_original" in df.columns:
    validation_cols.insert(5, "seq_length_original")

length_issues = df.loc[~df["lengths_match"], validation_cols]

print("Total rows:", len(df))
print("Rows with length / symbol issues:", len(length_issues))

if len(length_issues) > 0:
    length_issues.to_csv(VALIDATION_OUT, index=False)
    print(f"Validation issues saved to: {VALIDATION_OUT}")
    display(length_issues)
    raise ValueError("Length or symbol validation failed. Fix these rows before continuing.")

print("Length and symbol validation passed.")

In [ ]:
# 2A. Convert Chemical_Pattern into position-level annotations

In [ ]:
# ============================================================
# 2A. Position-level chemistry annotations
# ============================================================

SUGAR_MAP = {
    "M": "MOE",
    "C": "cEt",
    "d": "DNA",
}

def parse_aso_positions(row):
    """
    Converts one ASO row into one record per nucleotide position.
    """
    seq = row["Sequence"]
    pattern = row["Chemical_Pattern"]

    if len(seq) != len(pattern):
        raise ValueError(
            f"Length mismatch for ISIS={row['ISIS']}: "
            f"len(Sequence)={len(seq)}, len(Chemical_Pattern)={len(pattern)}"
        )

    records = []

    for i, (base, chem_symbol) in enumerate(zip(seq, pattern)):
        sugar = SUGAR_MAP.get(chem_symbol, "UNKNOWN")

        records.append({
            "row_id": row["row_id"],
            "ISIS": row["ISIS"],
            "position_0based": i,
            "position_1based": i + 1,
            "base": base,
            "chem_symbol": chem_symbol,
            "sugar": sugar,
            "is_DNA_gap": sugar == "DNA",
            "is_MOE": sugar == "MOE",
            "is_cEt": sugar == "cEt",
            "is_modified_sugar": sugar in {"MOE", "cEt"},
            "seq_length": len(seq),
        })

    return records


position_records = []
for _, row in df.iterrows():
    position_records.extend(parse_aso_positions(row))

pos_df = pd.DataFrame(position_records)

print("Position-level annotation shape:", pos_df.shape)
pos_df.head(25)

In [ ]:
# 2B. Parse linkage-level annotations

In [ ]:
# ============================================================
# 2B. Linkage-level annotations
# ============================================================

# Assumption:
# - "phosphorothioate" means all internucleotide linkages are PS.
# - "phosphodiester/phosphorothioate" means listed Linkage_Location positions are PO,
#   and all other linkages are PS.
# - Linkage positions are 1-based.
#
# For length n, there are n - 1 internucleotide linkages.
# Linkage 1 = between nt 1 and nt 2.
# Linkage 2 = between nt 2 and nt 3.

LINKAGE_LOCATION_IS_ONE_BASED = True

def parse_location_string(location):
    """
    Parses strings such as:
    '1?2?3?13?14/else'
    'else'
    ''
    
    Returns a set of integer positions as written in the file.
    """
    if pd.isna(location):
        return set()

    location = str(location).strip()

    if location == "" or location.lower() == "nan" or location == "else":
        return set()

    # Use only the part before '/else'
    first_part = location.split("/")[0].strip()

    if first_part == "" or first_part == "else":
        return set()

    # More robust than split("?"): extracts all integers
    return set(int(x) for x in re.findall(r"\d+", first_part))


def build_linkage_table(row):
    """
    Creates one row per internucleotide linkage.
    """
    aso_id = row["ISIS"]
    row_id = row["row_id"]
    n = int(row["seq_length"])
    linkage_type = str(row["Linkage"]).strip()
    linkage_location = str(row["Linkage_Location"]).strip()

    listed_positions = parse_location_string(linkage_location)

    records = []

    for linkage_1based in range(1, n):
        lookup_pos = linkage_1based if LINKAGE_LOCATION_IS_ONE_BASED else linkage_1based - 1

        if linkage_type == "phosphorothioate":
            linkage = "PS"
        elif linkage_type == "phosphodiester/phosphorothioate":
            linkage = "PO" if lookup_pos in listed_positions else "PS"
        else:
            linkage = "UNKNOWN"

        records.append({
            "row_id": row_id,
            "ISIS": aso_id,
            "linkage_position_1based": linkage_1based,
            "between_nt_1based": f"{linkage_1based}-{linkage_1based + 1}",
            "linkage": linkage,
            "is_PS": linkage == "PS",
            "is_PO": linkage == "PO",
            "is_unknown_linkage": linkage == "UNKNOWN",
        })

    return records


linkage_records = []
for _, row in df.iterrows():
    linkage_records.extend(build_linkage_table(row))

linkage_df = pd.DataFrame(linkage_records)

print("Linkage-level annotation shape:", linkage_df.shape)
linkage_df.head(25)

In [ ]:
# 2C. Linkage validation and summary features

In [ ]:
# ============================================================
# 2C. Linkage summary and validation
# ============================================================

linkage_summary = (
    linkage_df
    .groupby("row_id")
    .agg(
        n_linkages=("linkage", "size"),
        n_PS=("is_PS", "sum"),
        n_PO=("is_PO", "sum"),
        n_unknown_linkage=("is_unknown_linkage", "sum"),
    )
    .reset_index()
)

# Avoid duplicate columns if rerunning cells
cols_to_drop = [
    "n_linkages",
    "n_PS",
    "n_PO",
    "n_unknown_linkage",
    "expected_n_linkages",
    "linkage_count_valid",
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

df = df.merge(linkage_summary, on="row_id", how="left", validate="one_to_one")

df["expected_n_linkages"] = df["seq_length"] - 1
df["linkage_count_valid"] = df["n_linkages"] == df["expected_n_linkages"]

print("Linkage count validation:")
print(df["linkage_count_valid"].value_counts(dropna=False))

print("\nUnknown linkage count:")
print(df["n_unknown_linkage"].value_counts(dropna=False))

linkage_issues = df.loc[
    (~df["linkage_count_valid"]) | (df["n_unknown_linkage"] > 0),
    [
        "row_id",
        "ISIS",
        "Sequence",
        "Chemical_Pattern",
        "Linkage",
        "Linkage_Location",
        "seq_length",
        "n_linkages",
        "expected_n_linkages",
        "n_PS",
        "n_PO",
        "n_unknown_linkage",
        "linkage_count_valid",
    ]
]

if len(linkage_issues) > 0:
    linkage_issues.to_csv(VALIDATION_OUT, index=False)
    display(linkage_issues)
    raise ValueError("Linkage validation failed.")

print("Linkage validation passed.")

In [ ]:
# 2D. Chemistry-level summary features

In [ ]:
# ============================================================
# 2D. Chemistry summary features
# ============================================================

chem_summary = (
    pos_df
    .groupby("row_id")
    .agg(
        n_MOE=("is_MOE", "sum"),
        n_cEt=("is_cEt", "sum"),
        n_DNA=("is_DNA_gap", "sum"),
        n_modified_sugar=("is_modified_sugar", "sum"),
    )
    .reset_index()
)

# Avoid duplicate columns if rerunning cells
cols_to_drop = [
    "n_MOE",
    "n_cEt",
    "n_DNA",
    "n_modified_sugar",
    "frac_MOE",
    "frac_cEt",
    "frac_DNA",
    "frac_modified_sugar",
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

df = df.merge(chem_summary, on="row_id", how="left", validate="one_to_one")

df["frac_MOE"] = df["n_MOE"] / df["seq_length"]
df["frac_cEt"] = df["n_cEt"] / df["seq_length"]
df["frac_DNA"] = df["n_DNA"] / df["seq_length"]
df["frac_modified_sugar"] = df["n_modified_sugar"] / df["seq_length"]

# Internal consistency check
df["chem_count_sum_valid"] = (
    df["n_MOE"] + df["n_cEt"] + df["n_DNA"]
) == df["seq_length"]

print("Chemistry count validation:")
print(df["chem_count_sum_valid"].value_counts(dropna=False))

if not df["chem_count_sum_valid"].all():
    chem_issues = df.loc[
        ~df["chem_count_sum_valid"],
        [
            "row_id",
            "ISIS",
            "Sequence",
            "Chemical_Pattern",
            "seq_length",
            "n_MOE",
            "n_cEt",
            "n_DNA",
            "chem_count_sum_valid",
        ]
    ]
    chem_issues.to_csv(VALIDATION_OUT, index=False)
    display(chem_issues)
    raise ValueError("Chemistry count validation failed.")

df[
    [
        "ISIS",
        "Sequence",
        "Chemical_Pattern",
        "seq_length",
        "n_MOE",
        "n_cEt",
        "n_DNA",
        "n_PS",
        "n_PO",
        "frac_MOE",
        "frac_cEt",
        "frac_DNA",
    ]
].head()

In [ ]:
# Validation and save the outputs

In [ ]:
# ============================================================
# Final validation
# ============================================================

core_checks = [
    "valid_sequence",
    "valid_chemical_pattern",
    "lengths_match",
    "linkage_count_valid",
    "chem_count_sum_valid",
]

for col in core_checks:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

if not df[core_checks].all().all():
    raise ValueError("One or more core validation checks failed.")

# Optional: confirm expected total number of position rows
expected_position_rows = df["seq_length"].sum()
actual_position_rows = len(pos_df)

print("\nExpected position rows:", expected_position_rows)
print("Actual position rows:", actual_position_rows)

if expected_position_rows != actual_position_rows:
    raise ValueError("Position-level annotation row count does not match sum(seq_length).")

# Optional: confirm expected total number of linkage rows
expected_linkage_rows = (df["seq_length"] - 1).sum()
actual_linkage_rows = len(linkage_df)

print("\nExpected linkage rows:", expected_linkage_rows)
print("Actual linkage rows:", actual_linkage_rows)

if expected_linkage_rows != actual_linkage_rows:
    raise ValueError("Linkage-level annotation row count does not match sum(seq_length - 1).")

print("\nAll validations passed.")

In [ ]:
# ============================================================
# Save processed outputs
# ============================================================

df.to_csv(PROCESSED_OUT, index=False)
pos_df.to_csv(POSITION_OUT, index=False)
linkage_df.to_csv(LINKAGE_OUT, index=False)

data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "n_missing": [df[col].isna().sum() for col in df.columns],
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
})

data_dictionary.to_csv(DATA_DICTIONARY_OUT, index=False)

print("Saved files:")
print(PROCESSED_OUT)
print(POSITION_OUT)
print(LINKAGE_OUT)
print(DATA_DICTIONARY_OUT)